# 06 — Level 3: FACTS — Fairness of the Recourse
### (Kavouras, Sacharidis et al., NeurIPS 2023)

**Author:** Matteo

This is the core of the project. Levels 1-2 asked whether the model *predicts*
fairly. Here we ask the question of Von Kügelgen (2022) and FACTS (2023):

> *Even if prediction is fair, is the **recourse** — the set of changes DiCE
> asks an at-risk employee to make — equally achievable across groups?*

We load the recourse datasets produced in notebook 04 and compute, per
protected subgroup, the FACTS recourse-fairness notions.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import PROTECTED_ATTRIBUTES

BASE_PATH = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
OUT_DIR   = os.path.join(BASE_PATH, "data", "fairness")

recourse_rf  = pd.read_csv(os.path.join(OUT_DIR, "recourse_RF.csv"))
recourse_xgb = pd.read_csv(os.path.join(OUT_DIR, "recourse_XGB.csv"))
print(f"RF at-risk employees:  {len(recourse_rf)}")
print(f"XGB at-risk employees: {len(recourse_xgb)}")

BUDGET = None  # set later from the data (median cost) for "within budget"

## FACTS metrics, per subgroup

For a protected attribute we split the at-risk population into subgroups and,
for each, compute the FACTS notions:

| FACTS notion | Operationalisation here |
|---|---|
| **Equal Effectiveness** | share of the subgroup that obtains ≥1 valid CF |
| **Equal Cost of Effectiveness** | median L1 cost among those who get recourse |
| **Equal Effectiveness within Budget** | share obtaining recourse with cost ≤ budget |
| **Equal Choice for Recourse** | mean number of distinct CFs offered |

The **unfairness** for a notion is the max−min gap across subgroups.

In [ ]:
def facts_metrics(recourse_df, protected_attr, budget):
    rows = []
    for g in sorted(recourse_df[protected_attr].dropna().unique()):
        sub = recourse_df[recourse_df[protected_attr] == g]
        got = sub[sub["recourse_found"] == True]            # noqa: E712
        n = len(sub)
        rows.append({
            protected_attr:            g,
            "n_at_risk":               n,
            "EqualEffectiveness":      round(sub["recourse_found"].mean(), 3),
            "CostOfEffectiveness":     round(got["L1_cost"].median(), 3) if len(got) else np.nan,
            "EffectivenessWithinBudget": round((got["L1_cost"] <= budget).sum() / n, 3) if n else np.nan,
            "ChoiceForRecourse":       round(got["n_cf_found"].mean(), 2) if len(got) else np.nan,
        })
    return pd.DataFrame(rows)


def facts_gaps(table, attr):
    metrics = ["EqualEffectiveness", "CostOfEffectiveness",
               "EffectivenessWithinBudget", "ChoiceForRecourse"]
    out = {}
    for m in metrics:
        vals = table[m].dropna()
        out[m] = round(vals.max() - vals.min(), 3) if len(vals) > 1 else np.nan
    return out


# Choose a shared budget = global median recourse cost (so it's comparable)
all_costs = pd.concat([recourse_rf["L1_cost"], recourse_xgb["L1_cost"]]).dropna()
BUDGET = round(all_costs.median(), 3)
print(f"Budget (global median L1 recourse cost): {BUDGET}")

In [ ]:
facts_tables = {}   # (model, attr) -> table
facts_gap_records = []

for model_name, rdf in [("Random Forest", recourse_rf), ("XGBoost", recourse_xgb)]:
    print(f"\n{'='*64}\n{model_name} — FACTS recourse fairness\n{'='*64}")
    for attr in PROTECTED_ATTRIBUTES:
        tbl = facts_metrics(rdf, attr, BUDGET)
        gaps = facts_gaps(tbl, attr)
        facts_tables[(model_name, attr)] = tbl
        facts_gap_records.append({"model": model_name, "attribute": attr, **gaps})
        print(f"\n-- {attr} --")
        print(tbl.to_string(index=False))
        print(f"   UNFAIRNESS (max-min gap): {gaps}")

facts_gap_df = pd.DataFrame(facts_gap_records)
facts_gap_df.to_csv(os.path.join(OUT_DIR, "level3_facts_gaps.csv"), index=False)

## Visualisation 1 — Effectiveness & cost by subgroup (RF vs XGB)

In [ ]:
def plot_metric_by_group(metric, ylabel, fname):
    fig, axes = plt.subplots(1, len(PROTECTED_ATTRIBUTES),
                             figsize=(5 * len(PROTECTED_ATTRIBUTES), 4))
    for ax, attr in zip(axes, PROTECTED_ATTRIBUTES):
        rf_tbl  = facts_tables[("Random Forest", attr)]
        xgb_tbl = facts_tables[("XGBoost", attr)]
        x = np.arange(len(rf_tbl))
        ax.bar(x - 0.2, rf_tbl[metric],  0.4, label="RF",  color="#2196F3")
        ax.bar(x + 0.2, xgb_tbl[metric], 0.4, label="XGB", color="#FF9800")
        ax.set_xticks(x); ax.set_xticklabels(rf_tbl[attr], rotation=20, ha="right")
        ax.set_title(f"{metric} — {attr}")
        ax.set_ylabel(ylabel); ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=120)
    plt.show()

plot_metric_by_group("EqualEffectiveness", "share with recourse",
                     "level3_effectiveness.png")
plot_metric_by_group("CostOfEffectiveness", "median L1 cost",
                     "level3_cost.png")

## Visualisation 2 — Which features each subgroup is asked to change

Recourse can be "equally effective" but still ask different *kinds* of changes
of different groups. We aggregate the `changed_feats` column per subgroup.

In [ ]:
def feature_change_frequency(recourse_df, protected_attr):
    out = {}
    for g in sorted(recourse_df[protected_attr].dropna().unique()):
        sub = recourse_df[recourse_df[protected_attr] == g]
        counter = {}
        for feats in sub["changed_feats"].dropna():
            for f in str(feats).split(";"):
                if f:
                    counter[f] = counter.get(f, 0) + 1
        # normalise by subgroup size
        n = max(len(sub), 1)
        out[g] = {k: v / n for k, v in counter.items()}
    return pd.DataFrame(out).fillna(0)


# Example: top changed features by Gender under RF
freq_gender = feature_change_frequency(recourse_rf, "Gender")
top = freq_gender.assign(spread=freq_gender.max(axis=1) - freq_gender.min(axis=1)) \
                 .sort_values("spread", ascending=False).head(10).drop(columns="spread")
print("Top features with the largest cross-gender difference (RF):")
print(top.round(2).to_string())

ax = top.plot(kind="barh", figsize=(8, 5))
ax.set_title("Feature-change frequency by Gender (RF) — normalised")
ax.set_xlabel("share of subgroup asked to change this feature")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "level3_feature_changes_gender.png"), dpi=120)
plt.show()

## Master summary — all unfairness gaps, both models

In [ ]:
print("FACTS unfairness gaps (max-min across subgroups; 0 = perfectly fair):")
print(facts_gap_df.to_string(index=False))

### Reading Level 3
This is the project's thesis. If prediction looked fair (Level 1) but some
subgroup has lower **EqualEffectiveness** or higher **CostOfEffectiveness**,
we have demonstrated empirically — on the IBM HR data, with the team's own
RF/XGBoost models — Von Kügelgen's claim that *recourse fairness is
complementary to prediction fairness*. Comparing the gaps between RF and
XGBoost answers the second research question: **does the choice of model change
who bears the cost of staying?**